# Run routing experiments

Compare Jev, OpenAI Decisions, and Haiku as routers. This notebook creates routing experiments and runs metric evaluators on their saved decisions; it does not invoke solving models.

## Setup

With Python **3.12 or newer**, run these commands from the repository root. You only need to repeat installation when dependencies change.

```sh
python3.12 -m venv .venv
.venv/bin/python -m pip install -e ".[notebook]"
.venv/bin/python -m ipykernel install --user --name router-bench --display-name "Router Bench"
```

Put the selected providers' API keys in the root `.env` file. Existing environment variables take precedence. Keep `data/intern_flash_gpt5/` in the checkout; benchmark data and secrets are not bundled into the Python package. The editable install lets the package use those local files and picks up source edits without reinstalling.

## Run

Open this notebook in your editor or start JupyterLab from the repository root:

```sh
.venv/bin/python -m jupyterlab notebooks/run_routing_experiments.ipynb
```

Select the **Router Bench** kernel. After each kernel restart, run the imports below, then choose your experiment settings. Start Phoenix only if it is not already running. Dataset preparation can be skipped once the dataset and benchmark splits exist.

### Load imports and environment

Run this cell each session. It loads the root `.env` and configures the default local Phoenix URL and storage directory.


In [3]:
import os

from routing.upload_dataset import upload_dataset
from routing.pipeline import build_router, load_environment, run_pipeline_async as run_routing

load_environment()


### Start local Phoenix — only if needed

Skip this cell if Phoenix is already running (for example, through `./phoenix/start.sh`). Otherwise, run it once per kernel session to start local Phoenix with persistent storage in `phoenix/storage/`. Keep the kernel running while using this server.

For a server already running elsewhere, set `PHOENIX_BASE_URL` in the root `.env` and skip this cell.


In [4]:
import phoenix as px

os.environ["PHOENIX_BASE_URL"] = "http://127.0.0.1:6006"
phoenix_session = px.active_session() or px.launch_app(host="127.0.0.1", port=6006, use_temp_dir=False)


### Prepare the dataset — first use or missing splits

This cell uploads our dataset into Phoenix and creates test/development splits and benchmark splits.

Development and testing splits are fixed a **30-question development** and **12,416-question test** partitions

In [6]:
upload_dataset()


Reusing dataset: router-bench-intern-flash-gpt5-full-v1


'router-bench-intern-flash-gpt5-full-v1'

## Experiment settings

For subsequent experiments in the same session, start with the settings below. Reinitialize routers after changing router or prompt settings, then run the experiment cell.


## Choose routers

Edit `selected_routers` if you only wish to benchmark a subset of routers.

Supported options:
- **`jev`** — Jev (`jev-1.13.0`); requires `TYPESAFE_API_KEY`.
- **`openai-decisions`** — OpenAI Decisions (`gpt-6-luna`); requires `OPENAI_API_KEY`.
- **`haiku`** — Claude Haiku (`claude-haiku-5-5`); requires `ANTHROPIC_API_KEY`.


In [7]:
selected_routers = ["jev", "openai-decisions", "haiku"] # Update this list to test a subset of models

router_settings = {
    "jev": {"provider": "jev", "model": "jev-1.13.0"},
    "openai-decisions": {"provider": "openai_decisions", "model": "gpt-6-luna"},
    "haiku": {"provider": "anthropic", "model": "claude-haiku-5-5", "parameters": {"max_tokens": 4096}},
}


## Choose a system prompt

Our default system prompt is the following:

```
"You select a candidate language model to handle the current request.

You will receive:
- A list of candidates and descriptions of their capabilities and costs.
- The current request.

Assess the request's difficulty and the capabilities and costs described for each candidate. 
Select the candidate that best fits the request.

Return exactly one label:
Candidate A
Candidate B
Candidate C

Provide no explanation or additional text."
```

We have proided three variants that slightly bias the router towards different decisions. 
- **'balanced'**: "Balance cost and accuracy. Select the least expensive candidate that you expect to provide an accurate answer to the request."
- **'accuracy-first'**: "Prioritize highest accuracy. Only downgrade to a cheaper candidate when you are confident it can accomplish the task."
- **'cost-first'**: "Prioritize lowest cost. Only upgrade to a more expensive candidate when you do not believe a cheaper candidate can accomplish the task."

To try a prompt variant, set `system_prompt_variant` to `"default"`, `"balanced"`, `"accuracy-first"`, or `"cost-first"`.

Alternatively, you may override the entire system prompt with your own custom prompt. 

To use your own instructions, set `custom_system_prompt` to a nonempty string (including a triple-quoted multiline string). It overrides the selected variant's instructions. Only routing instructions are customizable.


In [8]:
system_prompt_variant = "default" # Set to "balanced", "accuracy-first", or "cost-first" to adjust behavior

custom_system_prompt = None  # Set to your own instruction string to override the default prompt.


## Initialize the selected routers

Each call builds the existing `ModelRouter` with the chosen instructions and the same anonymous capability descriptions. Initialization makes no routing requests.


In [9]:
routers = []
routers = [
    build_router(
        name=name,
        system_prompt_variant=system_prompt_variant,
        custom_system_prompt=custom_system_prompt,
        **router_settings[name],
    )
    for name in selected_routers
]


ANTHROPIC_API_KEY is set and takes precedence over the SDK's profile / federation auto-discovery; unset ANTHROPIC_API_KEY to use the auto-discovered credential.


## Select a split and benchmarks

Use `"development"` for the 30-question smoke test, or `"test"` for 12,416 questions across all benchmarks. 

Set `benchmarks_filter = None` to run all benchmarks, or select a subset, for example `benchmarks = ["aime", "gpqa"]`.

Supported benchmarks: `aime`, `livemathbench`, `livecodebench`, `swe-bench`, `gpqa`, `hle`, `mmlupro`, `simpleqa`, `arenahard`, `tau2`.

Benchmark selection stays within the chosen development/test partition. Each benchmark has three development questions. Phoenix splits such as `development/aime` and `test/aime` are created automatically by `upload_dataset()`, including when reusing an existing dataset. Each run loads the selected questions once and shares that exact dataset version across all routers.


In [10]:
split = "development"
benchmarks_filter = None  # None or a subset such as ["aime", "gpqa"].

## Run experiments

**This cell makes paid routing requests.** Routers run sequentially, with 16 concurrent questions per experiment. Rerunning creates new experiments and unique logs under `routing/results/`. Experiment IDs and log paths are printed; results are saved in Phoenix. Failures surface without deleting completed experiments or logs.

Only question prompts and the shared anonymous profiles reach routers. The private mapping remains Candidate A = Intern-S1, Candidate B = Gemini 2.5 Flash, Candidate C = GPT-5 Medium. Archived scores and published execution costs remain unchanged.


In [11]:
results = await run_routing(routers=routers, split=split, benchmarks=benchmarks_filter)


Loaded 30 development questions; dataset version: RGF0YXNldFZlcnNpb246MTU=
Running jev; log: /Users/caedinm/Projects/router-bench/routing/results/jev-6ba195dcc094419b9c68c22ff51d7ba0.jsonl
🧪 Experiment started.
📺 View dataset experiments: http://127.0.0.1:6006/datasets/RGF0YXNldDoz/experiments
🔗 View this experiment: http://127.0.0.1:6006/datasets/RGF0YXNldDoz/compare?experimentId=RXhwZXJpbWVudDozMw==


I1103 18:13:46.888972  319735 fork_posix.cc:71] Other threads are currently calling into gRPC, skipping fork() handlers
I1103 18:13:46.896011  322622 ev_poll_posix.cc:588] FD from fork parent still in poll list: fd(106, generation: 1)


running tasks |          | 0/30 (0.0%) | ⏳ 00:00<? | ?it/s

✅ Task runs completed.
Experiment completed: 30 task runs, 0 evaluator runs, 0 evaluations
Completed jev: experiment RXhwZXJpbWVudDozMw==
Running openai-decisions; log: /Users/caedinm/Projects/router-bench/routing/results/openai-decisions-4d5f0ab3e0ad4ed2ac7fae05143745f2.jsonl
🧪 Experiment started.
📺 View dataset experiments: http://127.0.0.1:6006/datasets/RGF0YXNldDoz/experiments
🔗 View this experiment: http://127.0.0.1:6006/datasets/RGF0YXNldDoz/compare?experimentId=RXhwZXJpbWVudDozNA==


running tasks |          | 0/30 (0.0%) | ⏳ 00:00<? | ?it/s

✅ Task runs completed.
Experiment completed: 30 task runs, 0 evaluator runs, 0 evaluations
Completed openai-decisions: experiment RXhwZXJpbWVudDozNA==
Running haiku; log: /Users/caedinm/Projects/router-bench/routing/results/haiku-a0f8a639b2a2450db97d91438c536659.jsonl
🧪 Experiment started.
📺 View dataset experiments: http://127.0.0.1:6006/datasets/RGF0YXNldDoz/experiments
🔗 View this experiment: http://127.0.0.1:6006/datasets/RGF0YXNldDoz/compare?experimentId=RXhwZXJpbWVudDozNQ==


running tasks |          | 0/30 (0.0%) | ⏳ 00:00<? | ?it/s

✅ Task runs completed.
Experiment completed: 30 task runs, 0 evaluator runs, 0 evaluations
Completed haiku: experiment RXhwZXJpbWVudDozNQ==


## Run Evaluators
Evaluators pair routing decisions with downstream results to determine how strong decisions actually were. This block may take some time to execute as Phoenix as evaluations must be run in sequence.

In [12]:
from evaluators.all_metrics import evaluate_experiment
from routing.pipeline import make_phoenix_client

# Evaluate the experiments returned by the most recent routing run.
evaluation_results = {}
evaluation_client = make_phoenix_client({}, asynchronous=True)
try:
    for router_name, experiment in results.items():
        experiment_id = experiment["experiment_id"]
        print(f"Evaluating {router_name}: {experiment_id}")
        evaluation_results[router_name] = await evaluate_experiment(
            evaluation_client, experiment_id
        )
        print(evaluation_results[router_name])
finally:
    await evaluation_client._client.aclose()


Evaluating jev: RXhwZXJpbWVudDozMw==
{'evaluated_runs': 30, 'skipped_failed_runs': 0, 'submitted_evaluations': 300, 'evaluation_errors': 0}
Evaluating openai-decisions: RXhwZXJpbWVudDozNA==
{'evaluated_runs': 30, 'skipped_failed_runs': 0, 'submitted_evaluations': 300, 'evaluation_errors': 0}
Evaluating haiku: RXhwZXJpbWVudDozNQ==
{'evaluated_runs': 30, 'skipped_failed_runs': 0, 'submitted_evaluations': 300, 'evaluation_errors': 0}


- After the evaluators finish, open each experiment in [Phoenix](http://127.0.0.1:6006). Navigate to Datasets and Experiments and find the recent experiments. Here you can view the individual routing outputs and evaluation metrics for each experiment, including any task or evaluation errors.

## Experiment summary

This table summarizes the experiments in `results` using their saved Phoenix metrics and configurations. 
- Mean Benchmark Accuracy is the question-weighted mean published downstream score across successful tasks.
- Total Cost per 1000 Prompts is the mean routing plus original published downstream cost in USD, multiplied by 1,000; questions flagged for invalid usage are excluded from cost across all candidates.
- Failed tasks are omitted and reported.

In [13]:
from math import isfinite
from statistics import mean
from urllib.parse import quote

import pandas as pd
from IPython.display import display

summary_client = make_phoenix_client({}, asynchronous=True)
summary_rows = []
experiment_summary_data = []
for router_name, experiment in results.items():
    experiment_id = experiment["experiment_id"]
    saved_experiment = await summary_client.experiments.get(experiment_id=experiment_id)
    metadata = saved_experiment["metadata"]
    response = await summary_client._client.get(
        f"v1/experiments/{quote(experiment_id, safe='')}/json", timeout=600
    )
    response.raise_for_status()
    runs = response.json()
    experiment_summary_data.append((router_name, metadata, runs))
    scores, costs = [], []
    missing_scores = missing_costs = failed_runs = excluded_costs = 0
    for run in runs:
        if run.get("error"):
            failed_runs += 1
            continue
        metrics = {metric["name"]: metric for metric in run["annotations"]}
        for metric_name, values in (("downstream_score", scores), ("total_system_cost", costs)):
            metric = metrics.get(metric_name, {})
            # The flag applies to the question across every candidate model.
            if metric_name == "total_system_cost" and metric.get("metadata", {}).get("invalid_usage"):
                excluded_costs += 1
                continue
            value = metric.get("score")
            if metric.get("error") or isinstance(value, bool) or not isinstance(value, (int, float)) or not isfinite(value):
                if metric_name == "downstream_score":
                    missing_scores += 1
                else:
                    missing_costs += 1
                continue
            values.append(value)
    summary_rows.append({
        "Model": metadata["router_config"]["model"],
        "Prompt Config": metadata["system_prompt_variant"],
        "Mean Benchmark Accuracy": mean(scores) if scores and not missing_scores else float("nan"),
        "Total Cost per 1000 Prompts": 1000 * mean(costs) if costs and not missing_costs else float("nan"),
    })
    if failed_runs or missing_scores or missing_costs or excluded_costs:
        print(f"{router_name} ({experiment_id}): {failed_runs} failed tasks; "
              f"{missing_scores} missing accuracy metrics; {missing_costs} missing cost metrics; "
              f"{excluded_costs} questions excluded from cost for invalid usage.")

experiment_summary = pd.DataFrame(summary_rows, columns=[
    "Model", "Prompt Config", "Mean Benchmark Accuracy", "Total Cost per 1000 Prompts"
])
display(experiment_summary.style.hide(axis="index").format({
    "Mean Benchmark Accuracy": "{:.2%}",
    "Total Cost per 1000 Prompts": "${:.4f}",
}, na_rep="N/A"))


Model,Prompt Config,Mean Benchmark Accuracy,Total Cost per 1000 Prompts
jev-1.13.0,default,46.67%,$28.5719
gpt-6-luna,default,36.67%,$12.5859
claude-haiku-5-5,default,50.00%,$31.2029


## Results by benchmark split

Run the experiment summary cell first. This table reuses its saved Phoenix data and groups questions by benchmark within the experiment’s development or test split. Mean Accuracy is the mean published downstream score for successful tasks. Total Cost is the sum of routing plus original published downstream cost in USD for eligible successful tasks, without scaling to 1,000 prompts. Invalid-usage questions are excluded from cost across candidates; failed tasks and exclusions are reported. Missing or errored required metrics produce N/A.


In [14]:
from collections import defaultdict

benchmark_summary_rows = []
for router_name, metadata, runs in experiment_summary_data:
    benchmark_runs = defaultdict(list)
    for run in runs:
        benchmark = run["reference_output"]["flash_dataset_id"]
        benchmark_runs[benchmark].append(run)
    for benchmark, group in sorted(benchmark_runs.items()):
        scores, costs = [], []
        missing_scores = missing_costs = failed_runs = excluded_costs = 0
        for run in group:
            if run.get("error"):
                failed_runs += 1
                continue
            metrics = {metric["name"]: metric for metric in run["annotations"]}
            for metric_name, values in (("downstream_score", scores), ("total_system_cost", costs)):
                metric = metrics.get(metric_name, {})
                if metric_name == "total_system_cost" and metric.get("metadata", {}).get("invalid_usage"):
                    excluded_costs += 1
                    continue
                value = metric.get("score")
                if metric.get("error") or isinstance(value, bool) or not isinstance(value, (int, float)) or not isfinite(value):
                    if metric_name == "downstream_score":
                        missing_scores += 1
                    else:
                        missing_costs += 1
                    continue
                values.append(value)
        benchmark_summary_rows.append({
            "Model": metadata["router_config"]["model"],
            "Prompt Config": metadata["system_prompt_variant"],
            "Benchmark Split": f'{metadata["split"]}/{benchmark}',
            "Mean Accuracy": mean(scores) if scores and not missing_scores else float("nan"),
            "Total Cost (USD)": sum(costs) if costs and not missing_costs else float("nan"),
        })
        if failed_runs or missing_scores or missing_costs or excluded_costs:
            print(f"{router_name}, {benchmark}: {failed_runs} failed tasks; "
                  f"{missing_scores} missing accuracy metrics; {missing_costs} missing cost metrics; "
                  f"{excluded_costs} questions excluded from cost for invalid usage.")

benchmark_summary = pd.DataFrame(benchmark_summary_rows, columns=[
    "Model", "Prompt Config", "Benchmark Split", "Mean Accuracy", "Total Cost (USD)"
])
display(benchmark_summary.style.hide(axis="index").format({
    "Mean Accuracy": "{:.2%}", "Total Cost (USD)": "${:.6f}",
}, na_rep="N/A"))


Model,Prompt Config,Benchmark Split,Mean Accuracy,Total Cost (USD)
jev-1.13.0,default,development/aime,100.00%,$0.155522
jev-1.13.0,default,development/arenahard,66.67%,$0.098174
jev-1.13.0,default,development/gpqa,33.33%,$0.012434
jev-1.13.0,default,development/hle,0.00%,$0.022272
jev-1.13.0,default,development/livecodebench,33.33%,$0.307165
jev-1.13.0,default,development/livemathbench,33.33%,$0.023847
jev-1.13.0,default,development/mmlupro,33.33%,$0.006344
jev-1.13.0,default,development/simpleqa,66.67%,$0.001866
jev-1.13.0,default,development/swe-bench,66.67%,$0.140203
jev-1.13.0,default,development/tau2,33.33%,$0.089333
